# Deploy to DataRobot — Registry → Custom Model Workshop

- **Author**: senkin.zhan@datarobot.com
- **Accelerator**: Deep learning for molecular SMILES: train and deploy on DataRobot — see [`README.md`](README.md)

Uploads the flat deployment bundle as a **custom inference model**, tests it, registers it
and deploys it.

What gets uploaded (everything lands flat in `/opt/code`, no subdirectories):

| Source | File | Role |
|---|---|---|
| `deploy/` | `custom.py` | DRUM hooks: `load_model` / `score` (and `fit`) |
| `deploy/` | `model_lib.py` | flattened `src/` inference library (regenerated by section 3 of this notebook) |
| `deploy/` | `model-metadata.yaml` | model name / type / target |
| `deploy/` | `requirements.txt` | extra pip deps (`torch==2.12.1`, rdkit, torch-geometric, PyYAML) |
| **`model/`** | `smiles_model.pth` | trained bundle written by `train.ipynb` (`state_dict`, `cfg`, `atom_map`, `tokenizer_vocab`, …) |

The trained artifact is read straight from `model/`, so `deploy/` holds code only — no
stale copy of the weights to keep in sync.

**Steps:** connect → rebuild `model_lib.py` from `src/` → check files → pick base environment → create/reuse custom model →
upload a new version → build the dependency image → custom model test → register + deploy →
score real SMILES through the deployment.

Re-running creates a **new version** of the same model (and replaces the model on the
existing deployment) rather than duplicating anything.


## 1. Connect

Credentials are read (in order) from:
1. `DATAROBOT_ENDPOINT` / `DATAROBOT_API_TOKEN` environment variables, or
2. `~/.config/datarobot/drconfig.yaml`.

In [1]:
import os
from pathlib import Path
import time

import datarobot as dr

print("datarobot SDK:", dr.__version__)

endpoint = os.environ.get("DATAROBOT_ENDPOINT")
token = os.environ.get("DATAROBOT_API_TOKEN")

try:
    if endpoint and token:
        client = dr.Client(endpoint=endpoint, token=token)
    else:
        # falls back to ~/.config/datarobot/drconfig.yaml
        client = dr.Client()
except dr.errors.ClientError as exc:
    raise RuntimeError(
        "DataRobot authentication failed. Refresh your API key at "
        "<app>/account/developer-tools and either update the `token:` line in "
        "~/.config/datarobot/drconfig.yaml or export DATAROBOT_ENDPOINT / "
        "DATAROBOT_API_TOKEN before starting the kernel."
    ) from exc

APP_ROOT = client.endpoint.rsplit("/api/v2", 1)[0]
print("endpoint  :", client.endpoint)
print("app root  :", APP_ROOT)

datarobot SDK: 3.19.0
endpoint  : https://app.jp.datarobot.com/api/v2
app root  : https://app.jp.datarobot.com


## 2. Settings

In [2]:
import yaml

# --- code layout (structural, not tunable) ---------------------------------
DEPLOY_DIR = Path("deploy").resolve()  # code: custom.py, model_lib.py, ...
MODEL_DIR = Path("model").resolve()  # trained artifact from train.ipynb
CONFIG_PATH = Path("config/config.yaml")

# --- everything below comes from config/config.yaml ------------------------
PROJECT_CFG = yaml.safe_load(CONFIG_PATH.read_text()) or {}
DEPLOY_CFG = PROJECT_CFG.get("deploy") or {}
PATHS_CFG = PROJECT_CFG.get("paths") or {}

# NOTE: custom.py inside the DRUM container loads "smiles_model.pth" by
# hard-coded name (it cannot read config.yaml). If you rename artifact_name
# here, update deploy/custom.py to match.
ARTIFACT_NAME = PATHS_CFG.get("artifact_name", "smiles_model.pth")

# --- how it shows up in the workshop ---------------------------------------
# The model name doubles as the deployment label. It is the one identifier that
# is stable across clusters (ids are not), so deploy.ipynb and predict.ipynb both
# read it from here.
MODEL_NAME = DEPLOY_CFG.get("model_name") or "smiles_deep_learning_regression"
MODEL_DESCRIPTION = DEPLOY_CFG.get(
    "model_description",
    "SMILES -> Tc regression (DMPNN / sequence NN) served through DRUM.",
)
TARGET_NAME = PROJECT_CFG.get("target", "Tc")
TARGET_TYPE = dr.TARGET_TYPE.REGRESSION
SMILES_COLUMN = (PROJECT_CFG.get("data") or {}).get("smiles_column", "SMILES")

# --- compute for the deployed model ----------------------------------------

INSTANCE_TYPE = str(DEPLOY_CFG.get("instance", "cpu")).lower()  # "cpu" | "gpu"
REQ_CPU_CORES = float(DEPLOY_CFG.get("cpu_cores", 1))
REQ_MEMORY_GB = float(DEPLOY_CFG.get("memory_gb", 1))
REQ_GPU_COUNT = float(DEPLOY_CFG.get("gpu_count", 0) or 0)
REPLICAS = DEPLOY_CFG.get("replicas") or None
RESOURCE_BUNDLE_ID = DEPLOY_CFG.get("resource_bundle_id") or None  # explicit override

if INSTANCE_TYPE not in ("cpu", "gpu"):
    raise ValueError(f"deploy.instance must be 'cpu' or 'gpu', got {INSTANCE_TYPE!r}")
if INSTANCE_TYPE == "gpu" and REQ_GPU_COUNT < 1:
    REQ_GPU_COUNT = 1

# --- base execution environment --------------------------------------------
# Any DataRobot drop-in env with torch preinstalled works; requirements.txt
# adds the rest on top of it.
BASE_ENV_SEARCH = DEPLOY_CFG.get("base_environment_search", "pytorch")
BASE_ENVIRONMENT_ID = DEPLOY_CFG.get("base_environment_id") or None

# --- step toggles -----------------------------------------------------------
# Defaults come from deploy.steps in config.yaml; environment variables still
# win at runtime (e.g. DR_RUN_MODEL_TEST=0).
STEPS_CFG = DEPLOY_CFG.get("steps") or {}


def _flag(env_name: str, cfg_key: str) -> bool:
    raw = os.environ.get(env_name)
    if raw is not None:
        return raw.strip().lower() in ("1", "true", "yes", "y")
    return bool(STEPS_CFG.get(cfg_key, True))


REBUILD_MODEL_LIB = _flag("DR_REBUILD_MODEL_LIB", "rebuild_model_lib")
CREATE_NEW_VERSION = _flag("DR_CREATE_NEW_VERSION", "create_new_version")
BUILD_DEPENDENCIES = _flag("DR_BUILD_DEPENDENCIES", "build_dependencies")
RUN_MODEL_TEST = _flag("DR_RUN_MODEL_TEST", "run_model_test")
REGISTER_AND_DEPLOY = _flag("DR_REGISTER_AND_DEPLOY", "register_and_deploy")
RUN_PREDICTION_TEST = _flag("DR_RUN_PREDICTION_TEST", "run_prediction_test")

IS_MAJOR_UPDATE = bool(DEPLOY_CFG.get("is_major_update", True))  # True -> v2.0, False -> v1.1
DEPENDENCY_BUILD_MAX_WAIT = int(DEPLOY_CFG.get("dependency_build_max_wait", 3600))

# --- test / deploy settings -------------------------------------------------
SCORING_SAMPLE_CSV = Path(DEPLOY_CFG.get("scoring_sample_csv", "input/test.csv"))
SCORING_SAMPLE_ROWS = int(DEPLOY_CFG.get("scoring_sample_rows", 20))
DEPLOYMENT_LABEL = MODEL_NAME

print("code dir  :", DEPLOY_DIR)
print("model dir :", MODEL_DIR)
print(
    f"compute   : instance={INSTANCE_TYPE}, cpu_cores>={REQ_CPU_CORES:g}, "
    f"memory>={REQ_MEMORY_GB:g}GB, gpu>={REQ_GPU_COUNT:g}, replicas={REPLICAS}"
)
print(
    "steps     :",
    {
        "rebuild_model_lib": REBUILD_MODEL_LIB,
        "new_version": CREATE_NEW_VERSION,
        "build_deps": BUILD_DEPENDENCIES,
        "model_test": RUN_MODEL_TEST,
        "register_deploy": REGISTER_AND_DEPLOY,
        "prediction_test": RUN_PREDICTION_TEST,
    },
)

code dir  : /home/notebooks/storage/deploy
model dir : /home/notebooks/storage/model
compute   : instance=cpu, cpu_cores>=2, memory>=6GB, gpu>=0, replicas=1
steps     : {'rebuild_model_lib': True, 'new_version': True, 'build_deps': True, 'model_test': True, 'register_deploy': True, 'prediction_test': True}


## 3. Rebuild `deploy/model_lib.py` from `src/`

DataRobot uploads land flat in `/opt/code`, so the `src/` package can't be shipped as-is.
This cell concatenates the inference subset of `src/` into a single `deploy/model_lib.py`
that `custom.py` imports — `src/` stays the source of truth.

* modules are concatenated in topological order (each may only use names defined earlier),
* intra-package imports (`from .pooling import ...`, `from src... import ...`) are stripped,
* every external import and every class/function body is preserved verbatim,
* legacy `from torch_scatter import scatter_add` lines are rewritten to a PyG fallback,
* an env prelude is prepended (`USER` / `HOME` / `TORCHINDUCTOR_CACHE_DIR`) because the DRUM
  container runs under a uid with no `/etc/passwd` entry and `torch._dynamo` calls
  `getpass.getuser()` at import time.

Set `DR_REBUILD_MODEL_LIB=0` to upload the existing `deploy/model_lib.py` untouched.


In [3]:
import re

# Topological order - each file may only reference names defined in earlier files.
MODEL_LIB_SOURCES = [
    "src/config.py",
    "src/utils.py",
    "src/data/smiles_data.py",
    "src/data/graph_features.py",
    "src/data/loader.py",
    "src/models/_scatter.py",
    "src/models/pooling.py",
    "src/models/sequence.py",
    "src/models/dmpnn.py",
    "src/training/trainer.py",
]

# Lines matching any of these are intra-package and must be stripped.
_INTRA_PKG_IMPORT_RE = re.compile(
    r"^\s*from\s+\.{1,2}\S*\s+import\s+.*$"  # from .X import Y, from ..X import Y
    r"|^\s*from\s+src(\.\S+)?\s+import\s+.*$"  # from src... import ...
    r"|^\s*import\s+src(\.\S+)?\s*(as\s+\w+)?\s*$"
)

# torch_scatter is a C++ extension whose wheels must match the exact torch build.
# Installing it inside the DataRobot container is fragile; modern PyG ships an
# equivalent `scatter` helper. src/ already imports from src/models/_scatter.py,
# so this rewrite only kicks in for legacy files importing torch_scatter directly.
_TORCH_SCATTER_IMPORT_RE = re.compile(r"^\s*from\s+torch_scatter\s+import\s+scatter_add\s*$")
_TORCH_SCATTER_SHIM = (
    "# Replaced `from torch_scatter import scatter_add` to drop the C++-extension dep.\n"
    "# A mismatched wheel fails with OSError (loading _scatter_cuda.so), not ImportError.\n"
    "try:\n"
    "    from torch_scatter import scatter_add  # noqa: F401 - fast path if installed\n"
    "except (ImportError, OSError):\n"
    "    from torch_geometric.utils import scatter as _pyg_scatter\n"
    "    def scatter_add(src, index, dim=0, dim_size=None):\n"
    "        return _pyg_scatter(src, index, dim=dim, dim_size=dim_size, reduce='sum')\n"
)

_ENV_PRELUDE = (
    "\nimport os as _os\n"
    '_os.environ.setdefault("USER", "drum")\n'
    '_os.environ.setdefault("HOME", "/tmp")\n'
    '_os.environ.setdefault("TORCHINDUCTOR_CACHE_DIR", "/tmp/torchinductor")\n'
    "del _os\n"
)


def _clean_module(path: Path) -> str:
    """Strip intra-package imports and rewrite legacy torch_scatter imports."""
    kept = []
    for line in path.read_text().splitlines():
        # Keep the source line's indentation. Emitting the replacement at
        # column 0 turns an indented import - one inside a `try:` or a
        # function body - into a SyntaxError in the flattened module.
        indent = line[: len(line) - len(line.lstrip())]
        if _INTRA_PKG_IMPORT_RE.match(line):
            kept.append(f"{indent}# stripped intra-package import: {line.strip()}")
        elif _TORCH_SCATTER_IMPORT_RE.match(line):
            for shim_line in _TORCH_SCATTER_SHIM.rstrip().split("\n"):
                kept.append(indent + shim_line if shim_line else "")
        else:
            kept.append(line)
    return "\n".join(kept)


def build_model_lib(sources=MODEL_LIB_SOURCES, root=Path("."), out_path=None) -> Path:
    """Concatenate the inference subset of src/ into a single flat module."""
    root = Path(root).resolve()
    out_path = Path(out_path) if out_path else DEPLOY_DIR / "model_lib.py"

    header = (
        '"""Auto-generated by deploy.ipynb (section 3). DO NOT EDIT BY HAND.\n\n'
        "Flat copy of the inference subset of src/ used by custom.py inside\n"
        "the DataRobot DRUM container. The src/ package remains the source of\n"
        "truth - re-run section 3 of deploy.ipynb after editing src/.\n\n"
        "Concatenated from (in order):\n"
    )
    header += "\n".join(f"  - {s}" for s in sources) + '\n"""\n'

    pieces = [header, _ENV_PRELUDE]
    for rel in sources:
        path = root / rel
        if not path.exists():
            raise FileNotFoundError(f"missing source module: {path}")
        pieces.append(f"\n# ===== {rel} =====\n\n")
        pieces.append(_clean_module(path))
        pieces.append("\n")

    out_path.write_text("".join(pieces))
    return out_path


if REBUILD_MODEL_LIB:
    lib_path = build_model_lib()
    compile(lib_path.read_text(), str(lib_path), "exec")  # syntax check
    print(
        f"rebuilt {lib_path} ({lib_path.stat().st_size:,} bytes) from {len(MODEL_LIB_SOURCES)} modules"
    )
else:
    print("skipped model_lib.py rebuild - uploading deploy/model_lib.py as-is")

rebuilt /home/notebooks/storage/deploy/model_lib.py (77,258 bytes) from 10 modules


## 4. Collect the files

Code comes from `deploy/`, the trained bundle from `model/`. Any `.pth` sitting in
`deploy/` is ignored on purpose — `model/` is the single source of truth for weights.


In [4]:
REQUIRED_CODE = ["custom.py", "model_lib.py"]

if not DEPLOY_DIR.is_dir():
    raise FileNotFoundError(f"{DEPLOY_DIR} not found — run this notebook from the project root.")

# code files: everything in deploy/ except weights and hidden files
code_paths = sorted(
    p
    for p in DEPLOY_DIR.iterdir()
    if p.is_file() and not p.name.startswith(".") and p.suffix != ".pth"
)
missing = [f for f in REQUIRED_CODE if f not in {p.name for p in code_paths}]
if missing:
    raise FileNotFoundError(f"missing required file(s) in {DEPLOY_DIR}: {missing}")

# trained artifact: read from model/, uploaded under its plain name
artifact_path = MODEL_DIR / ARTIFACT_NAME
if not artifact_path.is_file():
    raise FileNotFoundError(
        f"{artifact_path} not found — run train.ipynb first (it writes {ARTIFACT_NAME} there)."
    )

upload_paths = code_paths + [artifact_path]

total = 0
for p in upload_paths:
    size = p.stat().st_size
    total += size
    origin = "model/" if p == artifact_path else "deploy/"
    print(f"  {origin}{p.name:<24} {size / 1024:>10,.1f} KiB")
print(f"  {'TOTAL':<31} {total / 1024 / 1024:>10,.1f} MiB  ({len(upload_paths)} files)")

from datetime import datetime, timezone

print(
    "\nartifact mtime:",
    datetime.fromtimestamp(artifact_path.stat().st_mtime, timezone.utc).isoformat(),
)

# (local_path, path_inside_the_model) -> flat layout at /opt/code
upload_files = [(str(p), p.name) for p in upload_paths]

  deploy/custom.py                       7.3 KiB
  deploy/model-metadata.yaml             0.5 KiB
  deploy/model_lib.py                   75.4 KiB
  deploy/requirements.txt                1.0 KiB
  model/smiles_model.pth            1,871.4 KiB
  TOTAL                                  1.9 MiB  (5 files)

artifact mtime: 2026-09-02T11:28:00.141802+00:00


## 5. Pick the base execution environment

In [5]:
if BASE_ENVIRONMENT_ID:
    base_env = dr.ExecutionEnvironment.get(BASE_ENVIRONMENT_ID)
else:
    envs = dr.ExecutionEnvironment.list()
    matches = [e for e in envs if BASE_ENV_SEARCH.lower() in (e.name or "").lower()]
    if not matches:
        print("No match. Available environments:")
        for e in envs:
            print(f"  {e.id}  {e.name}")
        raise RuntimeError(f"No execution environment matching {BASE_ENV_SEARCH!r}")
    if len(matches) > 1:
        print("Multiple matches (using the first — set BASE_ENVIRONMENT_ID to override):")
        for e in matches:
            print(f"  {e.id}  {e.name}")
    base_env = matches[0]

BASE_ENVIRONMENT_ID = base_env.id
print(f"\nbase environment: {base_env.name}  ({BASE_ENVIRONMENT_ID})")


base environment: [DataRobot] Python 3.12 PyTorch Drop-In  (5e8c888007389fe0f466c72b)


## 6. Pick the compute (resource bundle)

`deploy.instance` / `deploy.cpu_cores` / `deploy.memory_gb` / `deploy.gpu_count` in
`config/config.yaml` are resolved to the **smallest DataRobot resource bundle that
satisfies all of them**. Set `deploy.resource_bundle_id` to pin one explicitly.


In [6]:
from datarobot.models.resource_bundle import ResourceBundle

GIB = 1024**3
all_bundles = [
    b
    for b in ResourceBundle.list()
    if "customModel" in b.use_cases and not getattr(b, "is_deleted", False)
]


def _fmt(b):
    gpu_mem = (b.gpu_memory_bytes or 0) / GIB
    gpu = f"{b.gpu_count or 0:g} x {gpu_mem:.0f}GB" if b.has_gpu else "-"
    return (
        f"  {b.id:<34} {b.name:<10} cpu={b.cpu_count:<5g} "
        f"mem={(b.memory_bytes or 0) / GIB:>6.1f}GB  gpu={gpu}"
    )


if RESOURCE_BUNDLE_ID:
    matches = [b for b in all_bundles if b.id == RESOURCE_BUNDLE_ID]
    if not matches:
        raise RuntimeError(f"resource_bundle_id {RESOURCE_BUNDLE_ID!r} not available")
    resource_bundle = matches[0]
    print("pinned by config.deploy.resource_bundle_id")
else:
    want_gpu = INSTANCE_TYPE == "gpu"
    candidates = [
        b
        for b in all_bundles
        if b.has_gpu == want_gpu
        and b.cpu_count >= REQ_CPU_CORES
        and b.memory_bytes >= REQ_MEMORY_GB * GIB
        and (b.gpu_count or 0) >= REQ_GPU_COUNT
    ]
    if not candidates:
        print(
            f"No {INSTANCE_TYPE.upper()} bundle satisfies "
            f"cpu>={REQ_CPU_CORES:g}, mem>={REQ_MEMORY_GB:g}GB, gpu>={REQ_GPU_COUNT:g}. Available:"
        )
        for b in sorted(all_bundles, key=lambda x: (x.has_gpu, x.memory_bytes, x.cpu_count)):
            print(_fmt(b))
        raise RuntimeError("no matching resource bundle — adjust `deploy:` in config/config.yaml")
    # smallest that fits: least memory, then least cpu, then least gpu memory
    # gpu_memory_bytes is None on some CPU bundles; mixing None with int
    # raises TypeError in Python 3.
    candidates.sort(
        key=lambda b: (
            b.memory_bytes or 0,
            b.cpu_count or 0,
            b.gpu_memory_bytes or 0,
        )
    )
    resource_bundle = candidates[0]
    print(
        f"{len(candidates)} {INSTANCE_TYPE.upper()} bundle(s) satisfy the request; smallest wins:"
    )
    for b in candidates[:5]:
        print(_fmt(b))

RESOURCE_BUNDLE_ID = resource_bundle.id
print("\nselected:")
print(_fmt(resource_bundle))

6 CPU bundle(s) satisfy the request; smallest wins:
  cpu.4xlarge                        4XL        cpu=2     mem=   6.0GB  gpu=-
  cpu.5xlarge                        5XL        cpu=2     mem=   8.0GB  gpu=-
  cpu.6xlarge                        6XL        cpu=2     mem=  10.0GB  gpu=-
  cpu.7xlarge                        7XL        cpu=2     mem=  12.0GB  gpu=-
  cpu.8xlarge                        8XL        cpu=2     mem=  14.0GB  gpu=-

selected:
  cpu.4xlarge                        4XL        cpu=2     mem=   6.0GB  gpu=-


## 7. Create (or reuse) the custom model in the workshop

In [7]:
existing = [m for m in dr.CustomInferenceModel.list() if m.name == MODEL_NAME]

if existing:
    custom_model = existing[0]
    print(f"reusing existing custom model: {custom_model.name} ({custom_model.id})")
else:
    custom_model = dr.CustomInferenceModel.create(
        name=MODEL_NAME,
        target_type=TARGET_TYPE,
        target_name=TARGET_NAME,
        description=MODEL_DESCRIPTION,
        language="python",
    )
    print(f"created custom model: {custom_model.name} ({custom_model.id})")

reusing existing custom model: smiles_deep_learning_regression (6a8d79a010c32b4d04545fb1)


## 8. Upload the files as a new version

`create_clean` starts from an empty file list, so the version contains **exactly** the
files in `deploy/` — no leftovers from previous uploads.

In [8]:
def _create_version_with_resources(files):
    """create_clean + resourceBundleId in one shot.

    The SDK's public create_clean() has no resource_bundle_id parameter, so the extra
    multipart field is injected through the internal _create(). If that internal API
    ever changes, fall back to create_clean() followed by a PATCH (which creates a
    minor version inheriting the files) so the deploy still gets the right compute.
    """
    try:
        return dr.CustomModelVersion._create(
            method="post",
            custom_model_id=custom_model.id,
            is_major_update=IS_MAJOR_UPDATE,
            base_environment_id=BASE_ENVIRONMENT_ID,
            base_environment_version_id=None,
            folder_path=None,
            files=files,
            extra_upload_data=[("resourceBundleId", RESOURCE_BUNDLE_ID)],
            network_egress_policy=None,
            maximum_memory=None,
            replicas=REPLICAS,
            required_metadata_values=None,
            training_dataset_id=None,
            partition_column=None,
            holdout_dataset_id=None,
            keep_training_holdout_data=None,
            max_wait=600,
        )
    except (AttributeError, TypeError) as exc:
        print("internal _create unavailable, falling back to create_clean + PATCH:", exc)
        clean = dr.CustomModelVersion.create_clean(
            custom_model_id=custom_model.id,
            base_environment_id=BASE_ENVIRONMENT_ID,
            is_major_update=IS_MAJOR_UPDATE,
            files=files,
            replicas=REPLICAS,
        )
        payload = {
            "isMajorUpdate": "false",
            "baseEnvironmentId": BASE_ENVIRONMENT_ID,
            "resourceBundleId": RESOURCE_BUNDLE_ID,
        }
        patched = client.patch(f"customModels/{custom_model.id}/versions/", data=payload).json()
        return dr.CustomModelVersion.get(custom_model.id, patched["id"])


if CREATE_NEW_VERSION:
    model_version = _create_version_with_resources(upload_files)
    print(f"created version {model_version.label}  ({model_version.id})")
else:
    versions = dr.CustomModelVersion.list(custom_model.id)
    if not versions:
        raise RuntimeError("no existing version — set CREATE_NEW_VERSION = True")
    model_version = versions[0]
    print(f"reusing version {model_version.label}  ({model_version.id})")

version_raw = client.get(f"customModels/{custom_model.id}/versions/{model_version.id}/").json()
print(
    f"resource bundle: {version_raw.get('resourceBundleId')} | replicas: {version_raw.get('replicas')}"
)
for item in version_raw.get("items", []):
    print(f"  - {item['filePath']}")

created version v9.0  (6a9808664b22d13f4507738e)
resource bundle: cpu.4xlarge | replicas: 1
  - custom.py
  - model-metadata.yaml
  - model_lib.py
  - requirements.txt
  - smiles_model.pth


## 9. Build the dependency image (`requirements.txt`)

Required before testing or deploying, because `rdkit` / `torch-geometric` are not in the
base image. Takes several minutes.

In [9]:
has_requirements = any(name == "requirements.txt" for _, name in upload_files)

existing_build = None
try:
    existing_build = dr.CustomModelVersionDependencyBuild.get_build_info(
        custom_model_id=custom_model.id,
        custom_model_version_id=model_version.id,
    )
except Exception:  # noqa: BLE001 - no build started yet
    existing_build = None

if existing_build is not None and existing_build.build_status == "success":
    print("dependency image already built for this version")
elif BUILD_DEPENDENCIES and has_requirements:
    try:
        build_info = dr.CustomModelVersionDependencyBuild.start_build(
            custom_model_id=custom_model.id,
            custom_model_version_id=model_version.id,
            max_wait=DEPENDENCY_BUILD_MAX_WAIT,
        )
        print("dependency build status:", build_info.build_status)
    except Exception as exc:  # noqa: BLE001 - surface the build log on failure
        print("dependency build failed:", exc)
        try:
            failed = dr.CustomModelVersionDependencyBuild.get_build_info(
                custom_model_id=custom_model.id,
                custom_model_version_id=model_version.id,
            )
            print(failed.get_log()[-5000:])
        except Exception as log_exc:  # noqa: BLE001
            print("(could not fetch build log:", log_exc, ")")
        raise
else:
    print("skipped dependency build")

dependency image already built for this version


## 10. Done — open it in the workshop

In [10]:
print(f"custom model id : {custom_model.id}")
print(f"version id      : {model_version.id}  ({model_version.label})")
print(f"base environment: {BASE_ENVIRONMENT_ID}")
print(
    f"compute         : {INSTANCE_TYPE.upper()} / {RESOURCE_BUNDLE_ID} "
    f"({resource_bundle.name}: cpu={resource_bundle.cpu_count:g}, "
    f"mem={resource_bundle.memory_bytes / 1024 ** 3:.1f}GB"
    + (f", gpu={resource_bundle.gpu_count:g}" if resource_bundle.has_gpu else "")
    + ")"
)
print()
print("Registry -> Model workshop:")
print(f"  {APP_ROOT}/registry/custom-model-workshop/{custom_model.id}/assemble")
print("(older UI route)")
print(f"  {APP_ROOT}/model-registry/custom-models/{custom_model.id}/assemble")

custom model id : 6a8d79a010c32b4d04545fb1
version id      : 6a9808664b22d13f4507738e  (v9.0)
base environment: 5e8c888007389fe0f466c72b
compute         : CPU / cpu.4xlarge (4XL: cpu=2, mem=6.0GB)

Registry -> Model workshop:
  https://app.jp.datarobot.com/registry/custom-model-workshop/6a8d79a010c32b4d04545fb1/assemble
(older UI route)
  https://app.jp.datarobot.com/model-registry/custom-models/6a8d79a010c32b4d04545fb1/assemble


## 11. Smoke-test the uploaded model

Uploads a small SMILES-only dataset and runs DataRobot's custom model test — the same
checks the *Test* tab in the workshop runs (image build, model loading, prediction on
a real payload, null-value imputation, side effects).


In [11]:
if RUN_MODEL_TEST:
    import pandas as pd

    sample = pd.read_csv(SCORING_SAMPLE_CSV)[[SMILES_COLUMN]].head(SCORING_SAMPLE_ROWS)
    sample_path = Path("smiles_scoring_sample.csv")
    sample.to_csv(sample_path, index=False)
    try:
        scoring_dataset = dr.Dataset.create_from_file(file_path=str(sample_path))
    finally:
        sample_path.unlink(missing_ok=True)
    print(f"scoring dataset: {scoring_dataset.name} ({scoring_dataset.id}), {len(sample)} rows")

    model_test = dr.CustomModelTest.create(
        custom_model_id=custom_model.id,
        custom_model_version_id=model_version.id,
        dataset_id=scoring_dataset.id,
        max_wait=3600,
    )
    print("overall status:", model_test.overall_status)
    for check, info in (model_test.detailed_status or {}).items():
        info = info or {}
        print(f"  {check:<26} {str(info.get('status')):<10} {str(info.get('message', ''))[:70]}")

    if model_test.overall_status not in ("succeeded", "warning"):
        print("\n--- test log (tail) ---")
        print(model_test.get_log()[-4000:])
        raise RuntimeError(f"custom model test failed: {model_test.overall_status}")
else:
    scoring_dataset = None
    model_test = None
    print("skipped custom model test")

scoring dataset: smiles_scoring_sample.csv (6a98086868c84d7b07077bb4), 20 rows
overall status: succeeded
  error_check                succeeded  
  null_value_imputation      succeeded  
  long_running_service       succeeded  
  side_effects               succeeded  
  prediction_verification_check skipped    
  performance_check          skipped    
  stability_check            skipped    
  chat_check                 skipped    


## 12. Register + deploy

Creates a registered model version in the Registry and deploys it to a prediction server.
On a re-run the existing deployment (same label) gets a **model replacement** instead of a
second deployment.


In [12]:
if REGISTER_AND_DEPLOY:
    registered = dr.RegisteredModelVersion.create_for_custom_model_version(
        custom_model_version_id=model_version.id,
        name=f"{MODEL_NAME} {model_version.label}",
        registered_model_name=MODEL_NAME,
        description=MODEL_DESCRIPTION,
    )
    print(f"registered model version: {registered.name} ({registered.id})")

    # A DataRobot Serverless prediction environment refuses a model package whose
    # build has not finished:
    #   422 Deploying model package without a completed build to DataRobot
    #       Serverless prediction environment is not supported.
    # Dedicated prediction servers are laxer, but waiting is correct either way -
    # if the build is already done this returns on the first poll.
    def _wait_for_build(version, timeout=1800, poll=10):
        deadline = time.time() + timeout
        status = version.build_status
        while True:
            if status == "complete":
                return status
            if status in ("failed", "error"):
                raise RuntimeError(
                    f"Model package build failed (build_status={status!r}). "
                    f"Check the model version in the Registry."
                )
            if time.time() >= deadline:
                if status in (None, "N/A", "n/a"):
                    raise RuntimeError(
                        f"Model package build never started (build_status={status!r}) "
                        f"after {timeout}s. Re-register the model to trigger it."
                    )
                raise TimeoutError(f"Model package build still {status!r} after {timeout}s.")
            print(f"  build_status={status!r}, waiting ...")
            time.sleep(poll)
            status = (
                dr.RegisteredModel.get(version.registered_model_id)
                .get_version(version.id)
                .build_status
            )

    print(f"build_status: {_wait_for_build(registered)}")

    existing_deployments = [
        d for d in dr.Deployment.list(search=DEPLOYMENT_LABEL) if d.label == DEPLOYMENT_LABEL
    ]
    if existing_deployments:
        deployment = existing_deployments[0]
        print(f"replacing model on existing deployment {deployment.id} ...")
        deployment.perform_model_replace(
            new_registered_model_version_id=registered.id,
            reason=dr.enums.MODEL_REPLACEMENT_REASON.OTHER,
            max_wait=1800,
        )
        print("model replaced")
    else:
        # Where to serve from. Two mutually exclusive routes, in priority order:
        #   1. a classic dedicated prediction server  -> default_prediction_server_id
        #   2. a prediction environment (serverless)  -> prediction_environment_id
        # Which of the two a given account/notebook token can see varies, so try
        # both rather than indexing blindly into a possibly-empty list.
        create_kwargs = {}

        pe_id = DEPLOY_CFG.get("prediction_environment_id") or os.environ.get(
            "DR_PREDICTION_ENVIRONMENT_ID"
        )
        ps_id = DEPLOY_CFG.get("prediction_server_id") or os.environ.get("DR_PREDICTION_SERVER_ID")

        if pe_id:
            create_kwargs["prediction_environment_id"] = pe_id
            print(f"using prediction environment from config: {pe_id}")
        elif ps_id:
            create_kwargs["default_prediction_server_id"] = ps_id
            print(f"using prediction server from config: {ps_id}")
        else:
            servers = dr.PredictionServer.list()
            if servers:
                create_kwargs["default_prediction_server_id"] = servers[0].id
                print(
                    f"using prediction server {servers[0].id} "
                    f"({getattr(servers[0], 'url', '?')})"
                )
            else:
                # No dedicated server visible - fall back to a prediction environment
                # that can actually host a custom model.
                envs = [
                    e
                    for e in dr.PredictionEnvironment.list()
                    if "customModel" in (e.supported_model_formats or [])
                ]
                # Prefer serverless, then a DataRobot-platform environment.
                envs.sort(
                    key=lambda e: {"datarobotServerless": 0, "datarobot": 1}.get(e.platform, 2)
                )
                if not envs:
                    raise RuntimeError(
                        "No prediction server and no customModel-capable prediction "
                        "environment is visible to this API token.\n"
                        "Fix by either:\n"
                        "  * setting deploy.prediction_environment_id (or "
                        "deploy.prediction_server_id) in config/config.yaml, or\n"
                        "  * exporting DR_PREDICTION_ENVIRONMENT_ID, or\n"
                        "  * using a token whose account has one "
                        "(Console -> Prediction environments)."
                    )
                create_kwargs["prediction_environment_id"] = envs[0].id
                print(
                    f"no prediction server visible; using prediction environment "
                    f"{envs[0].id} ({envs[0].platform}) - {envs[0].name}"
                )

        deployment = dr.Deployment.create_from_registered_model_version(
            model_package_id=registered.id,
            label=DEPLOYMENT_LABEL,
            description=MODEL_DESCRIPTION,
            max_wait=1800,
            **create_kwargs,
        )
        print(f"created deployment: {deployment.id}")

    print(
        f"\nConsole -> Deployments:\n  {APP_ROOT}/console-nextgen/deployments/{deployment.id}/overview"
    )
else:
    registered = None
    deployment = None
    print("skipped register + deploy")

registered model version: smiles_deep_learning_regression v9.0 (6a9808cd080306c62683a34d)
  build_status=None, waiting ...
  build_status='inProgress', waiting ...
build_status: complete
replacing model on existing deployment 6a8d8b32feab23af000adcd3 ...
model replaced

Console -> Deployments:
  https://app.jp.datarobot.com/console-nextgen/deployments/6a8d8b32feab23af000adcd3/overview


## 13. Final test — real predictions through the deployment

End-to-end check: send raw SMILES to the deployed model and compare the returned
predictions against the known `Tc` values from `input/test.csv`.


In [13]:
if RUN_PREDICTION_TEST and deployment is not None:
    import pandas as pd

    df = pd.read_csv(SCORING_SAMPLE_CSV).head(SCORING_SAMPLE_ROWS)
    job, scored = dr.BatchPredictionJob.score_pandas(deployment, df[[SMILES_COLUMN]].copy())
    try:
        print("batch job:", job.id, "|", job.get_status().get("status"))
    except Exception:  # noqa: BLE001
        print("batch job:", job.id)

    pred_cols = [c for c in scored.columns if "PREDICTION" in c.upper()]
    if not pred_cols:
        raise RuntimeError(f"no prediction column in response: {list(scored.columns)}")
    pred_col = pred_cols[0]

    out = df.copy()
    out["prediction"] = scored[pred_col].to_numpy()
    if TARGET_NAME in out.columns:
        out["abs_error"] = (out[TARGET_NAME] - out["prediction"]).abs()
        print(f"\nMAE over {len(out)} rows: {out['abs_error'].mean():.4f}")
    print()
    print(out.head(10).to_string(index=False))
else:
    print("skipped prediction test")

Streaming DataFrame as CSV data to DataRobot
Created Batch Prediction job ID 6a98096a29c618820b60e050
Waiting for DataRobot to start processing
Job has started processing at DataRobot. Streaming results.
batch job: 6a98096a29c618820b60e050 | COMPLETED

MAE over 20 rows: 0.0198

                                                                                                                          SMILES       Tc  prediction  abs_error
                                                                                                          *CC(*)C(=O)c1ccc(C)cc1 0.197500    0.204494   0.006994
                                                                                          *CCCCCCCCCNC(=O)C(CCCCCCCCCCCC)C(=O)N* 0.347000    0.345888   0.001112
                                                                                                                    *CC(*)CC(C)C 0.208333    0.214383   0.006050
                                                                             